# Libraries

In [1]:
library(Seurat)
library(ggplot2)
library(GEOquery)
library(Seurat)
library(dplyr)
library(glmGamPoi)

Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built with package ‘Matrix’ 1.7.5 but the current
version is 1.7.6; it is recomended that you reinstall ‘SeuratObject’ as
the ABI for ‘Matrix’ may have changed


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: Biobase

Loading required package: BiocGenerics

Loading required package: generics


Attaching package: ‘generics’


The following objects are masked from ‘package:base’:

    as.difftime, as.factor, as.ordered, intersect, is.element, setdiff,
    setequal, union



Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, is.unsor

# QC

## Checking metadata

In [ ]:
# read the merged Seurat object
merged_singlets <- readRDS("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/data/processed/merged_singlets.rds")
print(colnames(merged_singlets@meta.data))
print(table(merged_singlets$orig.ident))  

## Mitochondrial genes

In [ ]:
# mitochondrial gene percentage calculation
merged_singlets[["percent.mt"]] <- PercentageFeatureSet(merged_singlets, pattern = "^MT-")


## Visualization

In [ ]:
VlnPlot(merged_singlets,
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "orig.ident",
        pt.size = 0, ncol = 3)
ggsave("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/results/figures/qc_violin_by_sample.png", width = 14, height = 5, dpi = 300)

## Threshold

In [ ]:
# calculate the variance of nFeature_RNA and nCount_RNA for each sample 
FeatureScatter(merged_singlets, feature1 = "nCount_RNA", feature2 = "percent.mt")
FeatureScatter(merged_singlets, feature1 = "nCount_RNA", feature2 = "nFeature_RNA")

## GEO mata data mapping

In [ ]:
# check the distribution of hash.ID across samples to ensure that all cells are singlets
table(merged_singlets$orig.ident, merged_singlets$hash.ID)

# check the distribution of HTO_classification.global across samples to ensure that all cells are singlets
table(merged_singlets$orig.ident, merged_singlets$HTO_classification.global)

In [ ]:

# get GEO metadata for GSE300475
geo_meta <- getGEO("GSE300475", GSEMatrix = TRUE, getGPL = FALSE)

# extract the phenotype data (metadata) for the samples
pheno <- pData(geo_meta[[1]])

# show the columns that contain relevant information about the samples
pheno[, grepl("title|characteristics|source", colnames(pheno), ignore.case = TRUE)]
    
# key columns for review (the exact names may vary slightly; first view all)
print(colnames(pheno))
View(pheno[, grepl("title|characteristics|source", colnames(pheno), ignore.case = TRUE)])

# save the metadata to a CSV file for further review and processing
write.csv(pheno, "/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/data/raw/geo_sample_metadata_raw.csv", row.names = TRUE)

### build patient timepoint mapping

In [ ]:


merged_singlets <- readRDS("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/data/processed/merged_singlets.rds")

# the mapping table that links orig_sample, hashtag, patient_id, and timepoint
mapping_tbl <- tribble(
  ~orig_sample,              ~hashtag,  ~patient_id, ~timepoint,
  "scRNAseq PT1",            "C251",    "PT1",       "week1",
  "scRNAseq PT1",            "C252",    "PT1",       "week7",
  "scRNAseq PT1",            "C253",    "PT1",       "week15",
  "scRNAseq PT6",            "C251",    "PT6",       "week1",
  "scRNAseq PT6",            "C252",    "PT6",       "week7",
  "scRNAseq PT6",            "C253",    "PT6",       "week15",
  "scRNAseq PT7",            "C251",    "PT7",       "week1",
  "scRNAseq PT7",            "C252",    "PT7",       "week7",
  "scRNAseq PT7",            "C253",    "PT7",       "week15",
  "scRNAseq PT13",           "C251",    "PT13",      "week1",
  "scRNAseq PT13",           "C252",    "PT13",      "week7",
  "scRNAseq PT13",           "C253",    "PT13",      "week15",
  "scRNAseq PT15",           "C251",    "PT15",      "week1",
  "scRNAseq PT15",           "C252",    "PT15",      "week7",
  "scRNAseq PT15",           "C253",    "PT15",      "week15",
  "scRNAseq Week3",          "C251",    "PT6",       "week3",
  "scRNAseq Week3",          "C252",    "PT13",      "week3",
  "scRNAseq Week3_addition", "C251",    "PT1",       "week3",
  "scRNAseq Week3_addition", "C252",    "PT7",       "week3",
  "scRNAseq Week3_addition", "C253",    "PT15",      "week3",
  "scRNAseq PT15_add",       "C251",    "PT15",      "week1",   # replicate
  "scRNAseq PT15_add",       "C252",    "PT15",      "week7",   # replicate
  "scRNAseq PT15_add",       "C253",    "PT15",      "week15",  # replicate
  "scRNAseq PT11",           "C251",    "PT11",      "week3",
  "scRNAseq PT11",           "C252",    "PT11",      "week7",
  "scRNAseq PT11",           "C253",    "PT11",      "week15",
  "scRNAseq PT5",            "C251",    "PT5",       "week3",
  "scRNAseq PT5",            "C252",    "PT5",       "week7",
  "scRNAseq PT5",            "C253",    "PT5",       "week15",
  "scRNAseq Week1",          "C251",    "PT5",       "week1",
  "scRNAseq Week1",          "C252",    "PT11",      "week1",
)

# extract metadata from the merged singlets object
meta <- merged_singlets@meta.data

meta$hashtag_num <- case_when(
  grepl("C0251", meta$hash.ID) ~ "C251",
  grepl("C0252", meta$hash.ID) ~ "C252",
  grepl("C0253", meta$hash.ID) ~ "C253",
  TRUE ~ NA_character_
)

# for the samples that don't have a hashtag_num, assign based on hash.ID
meta$hashtag_num <- case_when(
  !is.na(meta$hashtag_num)              ~ meta$hashtag_num,
  meta$hash.ID == "Human-week1"          ~ "C251",
  meta$hash.ID == "Human-week7"          ~ "C252",
  meta$hash.ID == "Human-week15"         ~ "C253",
  TRUE ~ NA_character_
)

meta$cell_barcode <- rownames(meta)

meta_joined <- meta %>%
  left_join(mapping_tbl, by = c("orig_sample" = "orig_sample", "hashtag_num" = "hashtag")) %>%
  filter(!is.na(patient_id))  # check for any cells that didn't join properly

rownames(meta_joined) <- meta_joined$cell_barcode

# check the number of cells before and after the join
cat("number of cells before join:", nrow(meta), "\n")
cat("number of cells after join:", nrow(meta_joined), "\n")

# add the patient_id and timepoint columns to the Seurat object
merged_singlets <- AddMetaData(merged_singlets,
                                metadata = meta_joined[, c("patient_id", "timepoint")])

# final check of the distribution of patient_id and timepoint across the merged singlets
print(table(merged_singlets$patient_id, merged_singlets$timepoint))

saveRDS(merged_singlets, "/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/data/processed/merged_singlets_annotated.rds")

# QC final and replicate check

### Replicate tracking column for PT15

In [ ]:
merged_singlets <- readRDS("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/data/processed/merged_singlets_annotated.rds")

# replicate tracking column for PT15
merged_singlets$sequencing_run <- ifelse(
  grepl("PT15_add", merged_singlets$orig_sample),
  "PT15_replicate_run",
  "primary_run"
)



### QC of PT15

In [ ]:
# --- بررسی سریع: آیا نمونه‌ی تکراری PT15 با نمونه‌ی اصلی هم‌خوانی QC دارد؟ ---
merged_singlets[["percent.mt"]] <- PercentageFeatureSet(merged_singlets, pattern = "^MT-")

VlnPlot(subset(merged_singlets, patient_id == "PT15"),
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "sequencing_run", pt.size = 0, ncol = 3)
ggsave("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/results/figures/qc_PT15_replicate_check.png", width = 10, height = 4, dpi = 300)

## VlnPlot

In [ ]:
# --- final QC violin plots by patient and timepoint ---
VlnPlot(merged_singlets,
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "patient_id", pt.size = 0, ncol = 3)
ggsave("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/results/figures/qc_by_patient.png", width = 12, height = 5, dpi = 300)

VlnPlot(merged_singlets,
        features = c("nFeature_RNA", "nCount_RNA", "percent.mt"),
        group.by = "timepoint", pt.size = 0, ncol = 3)
ggsave("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/results/figures/qc_by_timepoint.png", width = 10, height = 5, dpi = 300)

## statistics abs for QC filtration

In [ ]:

# --- qc summary ---
summary_tbl <- merged_singlets@meta.data %>%
  group_by(patient_id, timepoint) %>%
  summarise(
    n_cells = n(),
    median_nFeature = median(nFeature_RNA),
    median_percent_mt = median(percent.mt),
    pct_above_15mt = mean(percent.mt > 15) * 100,
    .groups = "drop"
  )
print(summary_tbl)
write.csv(summary_tbl, "/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/results/tables/qc_summary_by_patient_timepoint.csv", row.names = FALSE)

### --- QC filtering ---

In [3]:
merged_singlets <- readRDS("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/data/processed/merged_singlets_annotated.rds")
merged_singlets$sequencing_run <- ifelse(
  grepl("PT15_add", merged_singlets$orig_sample),
  "PT15_replicate_run",
  "primary_run"
)
merged_singlets[["percent.mt"]] <- PercentageFeatureSet(merged_singlets, pattern = "^MT-")

# only keep cells with:
# - 200 < nFeature_RNA < 6000
# - nCount_RNA < 30000
# - percent.mt < 15
merged_filtered <- subset(
  merged_singlets,
  subset = nFeature_RNA > 200 & nFeature_RNA < 6000 &
           nCount_RNA < 30000 &
           percent.mt < 15
)

cat(" before filtration :", ncol(merged_singlets), "cells\n")
cat("after filtration:", ncol(merged_filtered), "cells\n")
cat("percentage of filtered out cells by PT15 line:\n")
print(table(merged_singlets$sequencing_run[merged_singlets$patient_id == "PT15"]))
print(table(merged_filtered$sequencing_run[merged_filtered$patient_id == "PT15"]))


 before filtration : 75467 cells
after filtration: 71653 cells
percentage of filtered out cells by PT15 line:

       primary_run PT15_replicate_run 
              3552              10757 

       primary_run PT15_replicate_run 
              3133              10482 


### --- SCTransform and PCA ---

In [4]:

#join the existing layers (based on raw orig_sample) first
merged_filtered[["RNA"]] <- JoinLayers(merged_filtered[["RNA"]])

# then split again based on actual patient_id
merged_filtered[["RNA"]] <- split(merged_filtered[["RNA"]], f = merged_filtered$patient_id)

merged_filtered <- SCTransform(merged_filtered, vars.to.regress = "percent.mt", verbose = FALSE, conserve.memory = TRUE)
merged_filtered <- RunPCA(merged_filtered, npcs = 30, verbose = FALSE)

# --- Integration based on patient_id (and implicitly sequencing_run as well, 
#since each patient_id layer contains data from both PT15 lines, Harmony will also adjust for the remaining run effect) ---
merged_filtered <- IntegrateLayers(
  object = merged_filtered,
  method = HarmonyIntegration,
  normalization.method = "SCT",
  verbose = FALSE
)

merged_filtered <- RunUMAP(merged_filtered, reduction = "harmony", dims = 1:30)
merged_filtered <- FindNeighbors(merged_filtered, reduction = "harmony", dims = 1:30)
merged_filtered <- FindClusters(merged_filtered, resolution = 0.6)


Will not return corrected UMI because residual type is not set to 'pearson'

Will not return corrected UMI because residual type is not set to 'pearson'

Will not return corrected UMI because residual type is not set to 'pearson'

Will not return corrected UMI because residual type is not set to 'pearson'

Will not return corrected UMI because residual type is not set to 'pearson'

Will not return corrected UMI because residual type is not set to 'pearson'

Will not return corrected UMI because residual type is not set to 'pearson'

Will not return corrected UMI because residual type is not set to 'pearson'



ERROR: Error: vector memory limit of 14.0 Gb reached, see mem.maxVSize()


### --- Visual inspection: Does the integration properly adjust for batch effects (patient and PT15 line)? ---

In [ ]:

DimPlot(merged_filtered, group.by = "patient_id", label = FALSE)
ggsave("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/results/figures/umap_by_patient.png", width = 7, height = 6, dpi = 300)

DimPlot(merged_filtered, group.by = "sequencing_run", label = FALSE)
ggsave("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/results/figures/umap_by_sequencing_run.png", width = 7, height = 6, dpi = 300)

DimPlot(merged_filtered, group.by = "seurat_clusters", label = TRUE)
ggsave("/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/results/figures/umap_by_cluster.png", width = 7, height = 6, dpi = 300)

saveRDS(merged_filtered, "/Users/mehranbeyki/portfolio-projects/tcga-brca-immune-checkpoint/data/processed/merged_integrated_clustered.rds")
